In [ ]:
# Data loading

import pandas as pd
import os
import numpy as np
import mne
import matplotlib
import matplotlib.pyplot as plt

# Load Excel file
excel_path = r'E:\TDBRAIN_participants_V2.xlsx'
df = pd.read_excel(excel_path)

# Filter records based on formal_status to include only those with 'MDD'
df_mdd = df[df['formal_status'] == 'MDD'].copy()

# Set the root directory path for preprocessing data
data_dir = r"E:\preprocessed_data_high gamma"

if not os.path.exists(data_dir):
    raise ValueError(f"data {data_dir} does not exist, check the path。")


available_folders = next(os.walk(data_dir))[1]

# Filter out records where participants_ID exists in the available data directory
# Assume the column name for patient ID in the Excel file is 'participants_ID'
# To ensure consistent comparison, convert all IDs to strings for matching
df_mdd['participants_ID'] = df_mdd['participants_ID'].astype(str)
available_folders = [str(folder) for folder in available_folders]

filtered_df = df_mdd[df_mdd['participants_ID'].isin(available_folders)].copy()
print("\n final MDD data：")
print(filtered_df)

In [ ]:
# Clustering analysis part

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

def extract_features(fif_path):

    try:
        epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
        data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
        mean_features = data.mean(axis=(0, 2))
        return mean_features
    except Exception as e:
        print(f"Error processing {fif_path}: {e}")
        return None

# Iterate through the filtered patients and extract features for each one
features_list = []
participant_ids = []

for idx, row in filtered_df.iterrows():
    pid = row['participants_ID']
  
    participant_dir = os.path.join(data_dir, pid)
    if os.path.exists(participant_dir):
        # Look for the session folder in each patient folder, the name starts with "ses-"
        session_dirs = [d for d in os.listdir(participant_dir) if d.startswith("ses-")]
        if not session_dirs:
            print(f"can not find session: {pid}")
            continue
 
        session_dir = os.path.join(participant_dir, session_dirs[0])

        # Construct fif file path, for example: E:preprocessed_data_high gammasub-xxxxxses-xeegpreprocessed-epo.fif
        fif_path = os.path.join(session_dir, "eeg", "preprocessed-epo.fif")
        if not os.path.exists(fif_path):
            print(f"can not find fif file {fif_path} for patient {pid}")
            continue
        feats = extract_features(fif_path)
        if feats is not None:
            features_list.append(feats)
            participant_ids.append(pid)
    else:
        print(f"file does not exist: {pid}")

if len(features_list) == 0:
    raise ValueError("no data for clustering！")


# Stack the extracted features into a matrix X (n_subjects, n_features)
X = np.vstack(features_list)
print(f"\nFeature matrix shape: {X.shape}")


# Elbow method (Inertia) & Silhouette in one figure

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib
import matplotlib.pyplot as plt

# If you really need a specific font, you can keep or remove this line
# matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei']

# ----- 1) Elbow method: compute inertia -----
distortions = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    distortions.append(kmeans.inertia_)

# ----- 2) Silhouette scores (k must be >= 2) -----
silhouette_scores = []
K_range2 = range(2, 11)

for k in K_range2:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"k = {k}, mean silhouette score = {score:.4f}")

# ----- 3) Plot both in ONE figure with two y-axes -----
fig, ax1 = plt.subplots(figsize=(8, 5))

# Left y-axis: Elbow (Inertia)
ax1.plot(K_range, distortions, 'bo-', label='Elbow (Inertia)')
ax1.set_xlabel('Number of clusters k')
ax1.set_ylabel('Inertia', color='b')
ax1.tick_params(axis='y', labelcolor='b')

# Right y-axis: Silhouette score
ax2 = ax1.twinx()
ax2.plot(K_range2, silhouette_scores, 'rs--', markerfacecolor='none',
         label='Silhouette score')
ax2.set_ylabel('Silhouette score', color='r')
ax2.tick_params(axis='y', labelcolor='r')

# Combine legends from both axes (in English)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

plt.title('Cluster number selection: Elbow vs Silhouette')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()


In [ ]:
# Visualization of clustering results

from sklearn.decomposition import PCA


best_k = 2  # based on the result above

# KMeans 
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=50)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('MDD Cluster Result')
plt.colorbar(scatter, label='Cluster Label')
plt.show()


In [ ]:
# Generate brain region heatmaps using representative samples from cluster centers (original EEG) while preserving temporal information

import seaborn as sns # type: ignore
from sklearn.metrics import pairwise_distances_argmin_min

# path
raw_data_dir = r"E:\preprocessed_data_high gamma"
target_filename = "preprocessed-epo.fif"

# Check whether cluster_labels and X exist
if 'cluster_labels' not in globals() or 'X' not in globals():
    raise NameError("Please run the clustering analysis first to ensure that 'cluster_labels' and 'X' are defined.")

# Calculate cluster centers
unique_labels = np.unique(cluster_labels)
cluster_representative_indices = {}
cluster_avg_amplitude = {}

for label in unique_labels:

    cluster_indices = np.where(cluster_labels == label)[0]
    cluster_samples = X[cluster_indices]
    
    
    center = cluster_samples.mean(axis=0, keepdims=True)
    
    
    closest_idx_in_cluster, _ = pairwise_distances_argmin_min(center, cluster_samples)
    closest_global_idx = cluster_indices[closest_idx_in_cluster[0]]
    
    cluster_representative_indices[label+1] = closest_global_idx

# Iterate through each cluster representative sample, extract its original EEG, and generate channel heatmap information
for label, idx in cluster_representative_indices.items():
    pid = participant_ids[idx]
    participant_dir = os.path.join(raw_data_dir, pid)
    if not os.path.exists(participant_dir):
        raise FileNotFoundError(f"can not find the directory：{participant_dir}")

    session_dirs = [d for d in os.listdir(participant_dir) if d.startswith("ses-")]
    found = False
    for ses in session_dirs:
        fif_path = os.path.join(participant_dir, ses, "eeg", target_filename)
        if os.path.exists(fif_path):
            try:
                epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
                data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
                ch_names = epochs.info['ch_names']

                # Preserve temporal information → Average absolute amplitude across channel dimension (μV)
                abs_amplitude = np.abs(data).mean(axis=(0, 2)) * 1e6
                cluster_avg_amplitude[label] = abs_amplitude
                print(f"cluster {label} representaive sample：PID={pid}, session={ses}")
                found = True
                break
            except Exception as e:
                print(f"loading fail {fif_path}: {e}")
    if not found:
        raise ValueError(f"cluster {label} finds no effective EEG file")

# bulid DataFrame
df_heatmap = pd.DataFrame(
    data=[cluster_avg_amplitude[label] for label in sorted(cluster_avg_amplitude)],
    index=[f"Subtype {label}" for label in sorted(cluster_avg_amplitude)],
    columns=ch_names
)

print("\n brain region heatmap data (μV)")
print(df_heatmap)

# heatmap
plt.figure(figsize=(15, 5))
sns.heatmap(df_heatmap, cmap='coolwarm', annot=True, fmt=".2f")
plt.title('Brain Channel Heatmap')
plt.xlabel('Channel')
plt.ylabel('Cluster')
plt.tight_layout()
plt.show()


In [ ]:
# Generate functional connectivity matrices using the representative sample of each cluster selected earlier (cluster_representative_indices)

import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns  # type: ignore
from scipy.stats import pearsonr

def compute_connectivity_matrix_single(fif_path):
    
    epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
    data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)

    # Concatenate the time points of all epochs
    concat_data = data.transpose(1, 0, 2).reshape(data.shape[1], -1)  # (n_channels, n_epochs*n_times)
    
    # Calculate the Pearson correlation coefficient matrix
    n_ch = concat_data.shape[0]
    mat = np.zeros((n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i, n_ch):
            r, _ = pearsonr(concat_data[i], concat_data[j])
            mat[i, j] = mat[j, i] = r
    return mat, epochs.info['ch_names']

# For each cluster representing the samples, construct the corresponding fif path and calculate connectivity
for label, rep_idx in cluster_representative_indices.items():
    pid = participant_ids[rep_idx]
    participant_dir = os.path.join(raw_data_dir, pid)
    session_dirs = [d for d in os.listdir(participant_dir) if d.startswith("ses-")]
    if not session_dirs:
        print(f"Representative sample {pid} of cluster {label} has no valid session, skipping")
        continue
    fif_path = os.path.join(participant_dir, session_dirs[0], "eeg", target_filename)
    if not os.path.exists(fif_path):
        print(f"The representative sample file of cluster {label} does not exist: {fif_path}")
        continue


    conn_mat, ch_names = compute_connectivity_matrix_single(fif_path)

    # plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(conn_mat,
                xticklabels=ch_names, yticklabels=ch_names,
                cmap='coolwarm', vmin=-1, vmax=1,
                square=True, cbar_kws={'label': 'Pearson r'},
                annot=False)
    plt.title(f'Subtype {label} Functional Connectivity Matrix')
    plt.xlabel('Channel')
    plt.ylabel('Channel')
    plt.tight_layout()
    plt.show()
